In [0]:
%cd /Workspace/Users/taymaz.ghaneh1@gmail.com/finance-timeseries-ml-platform

import site

# To Add the project's src directory as a site-package directory:
site.addsitedir("src")

In [0]:
%uv sync

In [0]:
%restart_python

In [0]:
%cd /Workspace/Users/taymaz.ghaneh1@gmail.com/finance-timeseries-ml-platform

import site

# To Add the project's src directory as a site-package directory:
site.addsitedir("src")

In [0]:
# To check whether the project package is installed
import importlib.metadata

try:
    print(
        "finance-timeseries-ml-platform:",
        importlib.metadata.version("finance-timeseries-ml-platform"),
    )
except importlib.metadata.PackageNotFoundError:
    print("finance-timeseries-ml-platform is NOT installed")

In [0]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import tensorflow as tf
import pyspark

import finance_ml

project_root = Path(
    "/Workspace/Users/taymaz.ghaneh1@gmail.com/"
    "finance-timeseries-ml-platform"
)

print("Python:", sys.version)
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("PySpark:", pyspark.__version__)
print("finance_ml:", finance_ml.__file__)

---
TESTING THE MODULES
---

In [0]:
import sys
import pytest
import os
import subprocess
from pathlib import Path

project_root = Path(
    "/Workspace/Users/taymaz.ghaneh1@gmail.com/"
    "finance-timeseries-ml-platform"
)

env = os.environ.copy()
env["PYTHONDONTWRITEBYTECODE"] = "1"

print("Python:", sys.executable)
print("Pytest:", pytest.__version__)
print("Project root:", project_root)
print("Project exists:", project_root.exists())


# Clear cached test/conftest modules so pytest re-imports
# the current versions from disk.
for _m in [
    k for k in list(sys.modules)
    if k == "conftest" or k.startswith("test_")
]:
    del sys.modules[_m]

exit_code = pytest.main(
    [
        "-q",
        "--assert=plain",
        "-p",
        "no:cacheprovider",
    ]
)

if exit_code != pytest.ExitCode.OK:
    raise RuntimeError(
        f"pytest failed with exit code {exit_code}"
    )




---
CHECKING GIT CHANGES
---

In [0]:
result = subprocess.run(
    ["git", "status", "--short"],
    cwd=project_root,
    text=True,
    capture_output=True,
)

print(result.stdout)

In [0]:
result = subprocess.run(
    ["git", "diff", "--stat"],
    cwd=project_root,
    text=True,
    capture_output=True,
)

print(result.stdout)

---
STARTING THE EXPERIMENT
---

In [0]:
spark.sql("SHOW TABLES IN finance_ml.gold").show(
    truncate=False
)

In [0]:
gold_df = spark.table("finance_ml.gold.account_features")

gold_df.printSchema()

In [0]:
print("Rows:", gold_df.count())

In [0]:
gold_df.selectExpr(
    "count(*) as n_rows",
    "count(distinct account_id) as n_accounts",
    "min(date) as min_date",
    "max(date) as max_date",
).show(truncate=False)

In [0]:
import os
import sys

import numpy as np
import tensorflow as tf

from finance_ml.models.experiment import run_lstm_experiment
from finance_ml.pipelines.model_dataset_pipeline import prepare_model_dataset


SEED = 42

np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("Python:", sys.version.split()[0])
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

In [0]:
# Preprocessing:
"""
temporal split
    ↓
feature-schema identification
    ↓
train-fitted median imputation
    ↓
categorical encoding
    ↓
final feature schema
    ↓
train-fitted MAD scaling
    ↓
model-column selection
    ↓
sequence construction
"""

from time import perf_counter

from finance_ml.pipelines.model_dataset_pipeline import (
    prepare_model_dataset,
)

start_time = perf_counter()

dataset = prepare_model_dataset(
    gold_df,
    sequence_length=7,
    stride=1,
    scaling_method="mad",
)

elapsed = perf_counter() - start_time

print(f"Model-dataset preparation completed in {elapsed:.2f} seconds.")

In [0]:
"""
Gold Spark DataFrame
        ↓
preprocessing
        ↓
train/validation/test split
        ↓
feature engineering
        ↓
sequence construction
        ↓
NumPy tensors
"""

print("Final features:", len(dataset["final_feature_columns"]))
print("Feature columns:")
for column in dataset["final_feature_columns"]:
    print(f"  - {column}")

print("\nSequence shapes:")
print("X_train:", dataset["X_train"].shape)
print("y_train:", dataset["y_train"].shape)
print("X_validation:", dataset["X_validation"].shape)
print("y_validation:", dataset["y_validation"].shape)
print("X_test:", dataset["X_test"].shape)
print("y_test:", dataset["y_test"].shape)

print("\nSplit boundaries:")
print("train_end:", dataset["train_end"])
print("validation_end:", dataset["validation_end"])

In [0]:
import numpy as np

for split in ("train", "validation", "test"):
    X = dataset[f"X_{split}"]
    y = dataset[f"y_{split}"]

    print(f"\n{split.upper()}")
    print(f"X shape: {X.shape}")
    print(f"y shape: {y.shape}")
    print(f"NaNs in X: {np.isnan(X).sum()}")
    print(f"Infs in X: {np.isinf(X).sum()}")
    print(f"Classes in y: {np.unique(y)}")

    classes, counts = np.unique(y, return_counts=True)

    for class_id, count in zip(classes, counts):
        print(f"  class {class_id}: {count}")

In [0]:
# To run the LSTM smoke test:
"""
Databricks Spark: Gold Spark table
      ↓
model dataset: prepare_model_dataset()
      ↓
NumPy tensors: 72,000 / 6,000 / 6,000 sequences
      ↓
LSTM training
      ↓
Artifact saving
      ↓
custom ValidationMacroF1 callback
      ↓
boundary trimming + overlap aggregation
      ↓
validation evaluation
      ↓
test evaluation
"""

import numpy as np
from finance_ml.models.experiment import run_lstm_experiment

SEED = 42

np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

from pathlib import Path

smoke_artifact_dir = (
    project_root
    / "artifacts"
    / "lstm_smoke_test"
)

result = run_lstm_experiment(
    x_train=dataset["X_train"],
    y_train=dataset["y_train"],
    metadata_train=dataset["train_metadata"],
    x_validation=dataset["X_validation"],
    y_validation=dataset["y_validation"],
    metadata_validation=dataset["validation_metadata"],
    x_test=dataset["X_test"],
    y_test=dataset["y_test"],
    metadata_test=dataset["test_metadata"],
    sequence_length=7,
    n_features=len(dataset["final_feature_columns"]),
    n_classes=4,
    lstm_units=(64,),
    dropout_rate=0.3,
    learning_rate=1e-3,
    boundary_width=2,
    epochs=1,
    batch_size=256,
    patience=0,
    verbose=1,
    artifact_dir=smoke_artifact_dir,
)

print("\nArtifact paths:")
for name, path in result.artifact_paths.items():
    print(f"{name}: {path}")

print("\nHistory keys:")
print(list(result.history.history.keys()))

print("\nTraining:")
print("Macro-F1:", result.history.history["macro_f1"][0])

print("\nValidation:")
print("Accuracy:", result.validation_evaluation["accuracy"])
print("Macro-F1:", result.validation_evaluation["macro_f1"])

print("\nTest:")
print("Accuracy:", result.test_evaluation["accuracy"])
print("Macro-F1:", result.test_evaluation["macro_f1"])

In [0]:
expected_artifacts = [
    "training_history.csv",
    "learning_curve_loss.png",
    "learning_curve_accuracy.png",
    "learning_curve_macro_f1.png",
]

for filename in expected_artifacts:
    path = smoke_artifact_dir / filename
    print(
        f"{filename}: "
        f"exists={path.exists()}, "
        f"size={path.stat().st_size if path.exists() else 0} bytes"
    )

In [0]:
# Running the baseline model:

from pathlib import Path

baseline_artifact_dir = (
    project_root
    / "artifacts"
    / "lstm_baseline"
)

baseline_result = run_lstm_experiment(
    x_train=dataset["X_train"],
    y_train=dataset["y_train"],
    metadata_train=dataset["train_metadata"],
    x_validation=dataset["X_validation"],
    y_validation=dataset["y_validation"],
    metadata_validation=dataset["validation_metadata"],
    x_test=dataset["X_test"],
    y_test=dataset["y_test"],
    metadata_test=dataset["test_metadata"],
    sequence_length=7,
    n_features=len(dataset["final_feature_columns"]),
    n_classes=4,
    lstm_units=(64,),
    dropout_rate=0.3,
    learning_rate=1e-3,
    boundary_width=2,
    epochs=50,
    batch_size=256,
    patience=5,
    verbose=1,
    artifact_dir=baseline_artifact_dir,
)

print("\nValidation:")
print(
    "Accuracy:",
    baseline_result.validation_evaluation["accuracy"],
)
print(
    "Macro-F1:",
    baseline_result.validation_evaluation["macro_f1"],
)

print("\nTest:")
print(
    "Accuracy:",
    baseline_result.test_evaluation["accuracy"],
)
print(
    "Macro-F1:",
    baseline_result.test_evaluation["macro_f1"],
)

print("\nArtifacts:")
for name, path in baseline_result.artifact_paths.items():
    print(f"{name}: {path}")

In [0]:
"""
artifacts/
├── lstm_baseline.keras
├── lstm_baseline_results.json
├── lstm_baseline_history.csv
├── lstm_baseline_loss.png
├── lstm_baseline_accuracy.png
└── lstm_baseline_macro_f1.png
"""

# artifacts are automatically saved by the run_lstm_experiment function.